# R.O.A.D. Historical HTR Training Pipeline

**Model:** Qwen2-VL-7B-Instruct  
**Hardware:** A100 80GB VRAM  
**Expected Training Time:** 6-8 hours  

This notebook:
1. Clones the repository
2. Downloads and extracts the image dataset
3. Installs dependencies
4. Runs training
5. Generates submission

## 1. Setup - Clone Repository

In [ ]:
# Clone the repo (or skip if already cloned)
import os

REPO_NAME = "ROAD"
REPO_URL = "YOUR_GITHUB_REPO_URL"  # Update this

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}
    print(f"✓ Cloned {REPO_NAME}")
else:
    print(f"✓ Repository {REPO_NAME} already exists")

%cd {REPO_NAME}

## 2. Download Image Dataset

In [ ]:
# Download images from Google Storage
import os

IMAGE_URL = "https://storage.googleapis.com/road-handwriting/images.zip"
IMAGE_DIR = "dataset/images"

if not os.path.exists(IMAGE_DIR) or len(os.listdir(IMAGE_DIR)) < 5000:
    print("Downloading images...")
    !wget -q --show-progress {IMAGE_URL} -O images.zip
    
    print("Extracting images...")
    !unzip -q images.zip -d dataset/
    !rm images.zip
    
    # Verify
    num_images = len([f for f in os.listdir(IMAGE_DIR) if f.endswith('.jpg')])
    print(f"✓ Downloaded {num_images} images")
else:
    num_images = len([f for f in os.listdir(IMAGE_DIR) if f.endswith('.jpg')])
    print(f"✓ Images already downloaded ({num_images} images)")

## 3. Install Dependencies

In [ ]:
# Install dependencies
%cd src/qwen2vl

!pip install -q -r requirements.txt

print("✓ Dependencies installed")

## 4. Verify Setup

In [ ]:
# Check CUDA and dataset
import torch
import pandas as pd
from pathlib import Path

print("=" * 60)
print("System Check")
print("=" * 60)

# CUDA
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"BF16 supported: {torch.cuda.is_bf16_supported()}")

# Dataset
print("\nDataset:")
train_csv = Path("../../dataset/Train.csv")
test_csv = Path("../../dataset/Test.csv")
image_dir = Path("../../dataset/images")

train_df = pd.read_csv(train_csv)
test_df = pd.read_csv(test_csv)
num_images = len(list(image_dir.glob("*.jpg")))

print(f"Train samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")
print(f"Images: {num_images}")

print("\n✓ Setup complete")
print("=" * 60)

## 5. Review Configuration

In [ ]:
# Display current config
import yaml

with open("config.yaml") as f:
    config = yaml.safe_load(f)

print("Current Configuration:")
print("=" * 60)
print(f"Model: {config['model']['name']}")
print(f"Batch size: {config['training']['batch_size']}")
print(f"Gradient accumulation: {config['training']['gradient_accumulation_steps']}")
print(f"Effective batch: {config['training']['batch_size'] * config['training']['gradient_accumulation_steps']}")
print(f"Epochs: {config['training']['epochs']}")
print(f"Learning rate: {config['training']['learning_rate']}")
print(f"LoRA rank: {config['training']['lora_r']}")
print(f"LoRA alpha: {config['training']['lora_alpha']}")
print(f"Augmentation: {config['augmentation']['enabled']}")
print("=" * 60)

## 6. Train Model

**Expected time:** 6-8 hours on A100 80GB  
**Expected VRAM:** 45-55 GB  

The training will:
- Load Qwen2-VL-7B-Instruct
- Apply LoRA fine-tuning
- Use augmentation on training images
- Evaluate every 100 steps
- Save best model based on eval loss

In [ ]:
# Start training
!python train.py

## 7. Generate Submission

Run inference on test set using the best checkpoint.

In [ ]:
# Run inference
!python inference.py

## 8. Check Submission

In [ ]:
# Verify submission format
import pandas as pd

submission = pd.read_csv("../../submission.csv")

print("Submission Preview:")
print("=" * 60)
print(submission.head(10))
print("\nSubmission Stats:")
print(f"Total predictions: {len(submission)}")
print(f"Empty predictions: {(submission['Target'] == '').sum()}")
print(f"Avg text length: {submission['Target'].str.len().mean():.1f} chars")
print("\n✓ Submission ready: ../../submission.csv")
print("=" * 60)

## 9. Sample Predictions (Optional)

In [ ]:
# Visualize some predictions
import matplotlib.pyplot as plt
from PIL import Image
import random

# Sample 3 random predictions
samples = submission.sample(3)

fig, axes = plt.subplots(3, 1, figsize=(15, 12))

for idx, (ax, (_, row)) in enumerate(zip(axes, samples.iterrows())):
    img_path = f"../../dataset/images/{row['ID']}.jpg"
    img = Image.open(img_path)
    
    ax.imshow(img)
    ax.set_title(f"Prediction: {row['Target']}", fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

## Next Steps

### For Better Performance:

1. **Error Analysis**
   - Review predictions on validation set
   - Identify common error patterns
   - Adjust augmentation strategy

2. **Hyperparameter Tuning**
   - Increase epochs to 7-10
   - Experiment with learning rate (1e-5 to 3e-5)
   - Try different LoRA ranks (32, 64, 128)

3. **Ensemble**
   - Train TrOCR-large
   - Ensemble Qwen + TrOCR predictions
   - Test-time augmentation (multiple passes)

4. **Post-processing**
   - Historical spelling correction
   - Language model filtering
   - Common phrase detection